# 30D same-H100 runtime benchmark: HCN vs NN4MGF

这是从仓库 `benchmarks/run.py` 提取出的**单文件 Jupyter Notebook**，只做 30D 的 same-hardware runtime / peak-memory benchmark。

- **HCN**：30D product-form，seed 12345，50,000 updates，1024/128/128 batches，64M base inputs。
- **NN4MGF**：公开训练代码，30D，300,000 updates，`batch_size=16384`、`train_freq=2`，即每个 optimizer update 实际处理 32,768 个 base inputs；3 个 100k LR stages。
- 计时只覆盖训练 updates（sampling + forward + losses + backward + optimizer + upstream per-update logging），**不把 checkpoint 保存、tail inversion、raw/clipped evaluation 算进 runtime**。
- 所以 **raw vs clipped 对 runtime benchmark 没有影响**。最终 accuracy 可以另外在本地 checkpoint 上评估。

内嵌源文件 SHA256：
- HCN 30D notebook: `3896e23997cfc939ee4871833591a3714e0a79791086ad700fea56240a5b2018`
- NN4MGF `fit_mgf.py`: `46d123430d459a2dfee064ace4742bdf213fde024c49bdd4da317f1a549364e7`
- NN4MGF 300k schedule: `172d06a525894b8f1889cc863016b7750343403a54caf35c447b368824333d53`

推荐流程：先 `MODE='profile'` 跑 10 warm-up + 200 measured updates；确认显存和预计总时间后，重启 kernel，把 `MODE='full'` 再 Run All。


In [ ]:
# =========================
# 1. 只需要改这里
# =========================
MODE = "profile"   # "smoke", "profile", "full"
OUTPUT_ROOT = "benchmark_30d_same_gpu"

RUN_HCN = True
RUN_NN4MGF = True

PROFILE_WARMUP = 10
PROFILE_MEASURED_UPDATES = 200
CPU_THREADS = 4

assert MODE in {"smoke", "profile", "full"}
print({"MODE": MODE, "RUN_HCN": RUN_HCN, "RUN_NN4MGF": RUN_NN4MGF})


## 2. 环境检查

服务器只需要有 CUDA PyTorch。NN4MGF 还需要 `pandas`, `tqdm`, `matplotlib`, `mpmath`, `numpy`。如果缺包，再取消下一格最上面的 `%pip install ...` 注释。

In [ ]:
# 如服务器缺少这些小依赖，再取消下一行注释；不要在这里重装 torch。
# %pip install -q pandas tqdm matplotlib mpmath numpy

import os, sys, time, json, csv, math, random, statistics, hashlib, types, ast
from pathlib import Path
from datetime import datetime
import subprocess
import numpy as np
import torch
import pandas as pd

if MODE in {"profile", "full"} and not torch.cuda.is_available():
    raise RuntimeError("profile/full 必须在 CUDA GPU 上运行。")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GiB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)
    try:
        print(subprocess.check_output([
            "nvidia-smi", "--query-gpu=name,uuid,driver_version,memory.total",
            "--format=csv,noheader"
        ], text=True).strip())
    except Exception as e:
        print("nvidia-smi query skipped:", e)

torch.set_num_threads(CPU_THREADS)


In [ ]:
# =========================
# 3. 内嵌原始源代码（无需额外 .py 文件）
# =========================
import base64, zlib

_HCN_B64 = '...'
_NN4MGF_B64 = 'eNrtfe1y40aS4H89RQ0VtwYkkE2ybd8Mx5zdHrs94YtujUPdc54JLY2ASFCEGwRoAJQo9fb83ue8J7nMrO9CgaRk+XZudx2OFoGqyqrKysqvykpk601ZNaysTzL+a500K/l7Tg/Lqlyz5n6TFTdMFHyd5HlynacRe5PVDf7bpFWSR+zPmyYrC/z1frvJUwV0Y4IttuvNPUtqVmzkq01SLOAF/L9ZyHdNWc1X1sOgKKhZ4b4dLLfFnPeMFb61y0sY1Brf0w8xH10yyKu4nq/SxTZPKznD17tNWaRFkyX5m0uzyc/bpM4qGC68EXXflddl/rq4yYrUwOImL5s8ux5s7vEXTS1vBKSfF6ox/j45+bpcA7p23xZsqpB7dTXnb2cRk79OTvgo6rSJF+ky2ebwF9YmDfj7ZV4mzZefhycn8zypARPltsrS6ts0abZVWgeAqrclTjScnDD4r9fr0V9Rjy1FRbYsK9kpy4rNtmEP7F+zgn3948fFJxbUq2STTlhwES3CcHBCMPrs+7TqLwDVEsqE1VnxYl4C5gt2mQYP8YcwYt+t6YdoIykGSCCr7rI61WMIZKssPKe/P4UhH1hV1jX1lBVIeLT0NQFkLMhw1GtYu3TB6mSZ5vfsLmtWJczhuiqTxTypG6TlVTL/UId87H/eNjDHCatSGIkaQLlkNFGcJ72Noc/QQhwsAovjrMiaOA7ECBir03wZqafFBMepn4H+Y40iKIJFfzmOqOyUbQQSr7eLm7T5PVtvaxh4ytLbtFAgluusmDBabWg8HHwRGUXJLt7pwi8Hw0iWnQJh7tiySn8mPNJsNwlQIaxsUVbrJM8eAGvzsqwWtQ3xXkMcjTVIB2K2Tm66IcKKLpu0YKvsZpVWoepgW6exXP4Juy7LHHp5X21TA4dI5BOxB+kBqlg0z+sKuqZF2AIqg3Cglkd3mC2tRWD/g43ZbwCPujH+VyVIjv87ybfp66oqq6BnNTIXphdaaz9YwOgW9iur7dTq3664SvKlU4G9eMHGdi0TZ1DbfDw50euNK7NNi3kGQJIqRXoFNrXADdleHlizq/4oGs300kPzOt4pXOflTb1J5mmABBgJWov0uCO+UFP6N2SnTiuUAvg0GhIAYAbWGwQWdkJzBnW/d1D3zzmo+yMGReVVegPyMK3i6+1yCcTXE/jrRRKTIS5KQHCOa3uv2t5bbY01Bkq5Bt4N3EoRRJB99VPYpj8sR+ILFuwM/+mzEbBUJC7cEhZJpTn8MzxRTE6yvwCBhaz/B2RcE2MUsNkYlGuej/xgx86tZ1i0P7DPzwSJj8+8uwAgYU9QQewlGOvn8I9aAmv22C+O2hA2Sxbs4gy63oHIwB7HZ55NZfWHEGSHGlPQ8bij4yoFIIUa67mCYqCsrO6SakEYi9hDjBtOsrD3aVGXFeHRfKERKgUM/iebSoHcUF0ufSPQLOrtGjZxkgM/X9yb+7opxYYGprobOGOvhbjT0BwZ5w4DKiBf46MZkGRUZTCIFLg+lMtV06ja6UbUIVIxDl1VuNcVSICoCsYy9+E/JRsVX8S3RiUaU831kogvmBrENRLcYAjLSdt7k8Gv3WBb1MAf04c06I9CudRirw5us/QuGEUM/odSPVwfqPt9oO5dUC6tE1HWiqPNkyawRNGVUPuyIthdw5qLamVtPmLpvV0Kj7PIggSdTYEkJI4jvhn15EJYI0Ij0gNOhFdki7NAVDVGj3IUdNu2QAJ6Env3K9AXHLFq7Rw+8/ZKS0iGhmdohkhHfSImoP06tOkA2yqEXs0M0GkyX1ExEH21ZjfZLQKD2XEGgZupWLC7FHZaAWvAylvBXGrcqFhR8YazYCzRoeQSKkAoRME8uEkDjoDQnjzW+UnXQSY1EsLFrSqngvx6dzWJWDZDloa/fpqFDrnxVTKIVEHwESuB7SB2DY2mh1JBCi/v8DimB8lmkxaLQFPi5joMj26AtEoN/Iuod4V+HQlSVhTqWxqXXWtAVxb9RUZ/MwVZWlFvyps/vf32Im1a9tNT1P/lMl5LtV+qrWJnxilI8kVMYlRVOAXb4HQ0/m10atZeZYtFWphVsYoewjzJ0/j6Pn5Iq7JDp97FlhXR/wLUerMw2ZkmhlF27zQcfemUmi3tQttwGaX9cbflMmqZNYYRMjahIu+5LrfFIqnuOQ7VlL9NQJHRNe1aLqZbWH5es6PLNLAWC7UQ89mu+tf47XcXkfr96q9QndYx4itm1/6bUftvovY9r33frt3GojAs7JfOdARS6K9dtIS92MSaUKGWfrCECC3ShJ2SuBg7ZhiSDDTFP857JJcpUEn79f10bLxGXdYLs/+5D+LQD3E0GJpi6t0K7KkFG33jOk+Az24cRKDq6bpibKk8Hdli2lRUp8gxotb4p2Tu+CYgzDLfJIRxZBWZknvq7Be1Cbi9ows0VZPxGEvVUSrRS/hfmgyWIvc1Vs+KpEn7WbEAbZaIakHexaLOFilrVinw68V2ni5ecN8TAAElFVQAh/Ykx6TWoN0Xg9fyMVhENke1BvFHQdHuECbsbpWBjiApnknHIhnPoBjkaVIVUPGfDWDfgkzHMatGRdrcldUHROxCTurtn1CARHZF3jupAjVXNj4OIzYYDGDsn4we7lZphUpVdpOhq2y5zXONE5aBDiNYC2gTaKuWYAzkJYg5xIsuwpoAY9UMzM3XsfftDUOV7ApepHN9ps1lDezXtHGAOwpycejn3F62x4/UAn8+9QzmxCYjPgBYND6bd+QxQc+vvUeh6E1WwPoHVheRn9nZboLIhfQue/OXoP1aduCH+EzQTTBjB5JyQc7VNkVluKmy6y3uA7CtLlP04RoWg4PO6yypOSa/T6pknYKqLbQ8FGc16phOr6G2mWNlvUqz+WiLeVcpTTd44JZm3xCaIXshtgmXm3ZZnyF7VypMZoIim7RvSFQN6m8mqL/5QNmKJzfgg10FAjgzp03oRgasuP7zuQ3IfAul9yDyOArangFqEzl7MySHQZdv4JsDvoFvfL6Bh3iZkz6nHATC+ATUfwPmcihZYHBxtqBnMQ2tGKLOLqEI0RNwsKFB/hLC3hlJe0PDNK3hb4RxJC3iFjB30Q3Tlha5Su5i4RSi2ab7fENg7y1204uySH+95YYebObZZ9ght+u1mDIkdE1CzWmDCvQLNuwvpC/pBffE9GEooXxnCr1sDvIOOjmiB4siPcv/HEQI6wmQvWdg5DHQvNCj9NSS7Hx7mA9lL82YoxiHTMlScQSnuDDXFfAkC1RX1CLm26oCOcWP5ywrMltooznhjgag3UV6C3ifCuzwp5C1/oNhLAzftBDGyhlrqlyB6MwDxYHnqmJt6OLXoCkDA7cDx8verq/dIMNwkO7wIJl7ruRmbY2GL0OXbrjJ0jkd0FwZw1DVZ9ZyvQz18eW10igXpjpZgwC0ypKmSfCwGX2zKWh89z7qOlbXgXqwhXHP4LZtu4/aJ1hYXR5cbaryFlTTBaqXhcfQm6K7oGfgxnD5ceLK6pjv7gDgetxX2Juyl0FpDZAhRMgvqIGfJqVuII9pihvbj9S25nRX8C9S0FPAanwOClJMW+eB7a4kvbUhpbkDa9QBi9ciXnU1nOEp5IW/pnc9l7SgK1C4+EH1RwXsU8RgdOkcT8GDjxeforDnG6UPlYcpJyEXRVKhnzdBE5hTgUsrJjlJBtK2IYgUWMdO9VkSe7owHw+wkkOtNWMZKcaCXKWtBRwaI2cp0uFp1jbwxUfKzV3L4UmNDWfnHj7Lh2NZJ6HHZbFI5xkZjtU2h7WkaAo0GA1OVDvjKrdKu1LWUmAM+Zhxja2xvIK1b7Jb6P/mpkpvksa07nl3qutBvV2jrJmO9kqaVqfj0IGI/54bVsoj/lMQHZC2Yg/v0D8/nIxmERMPo8l4FnaDHLXURmjXeYj4S3VEVz3EowjlV/CtP6qIrsMiK5aoS5JfRJwbeyTHk1Q4zrKVSqWtwQcDTcmdquBRqTkIQo19VNX2tdq8D5SqJhmqNSVrNc6zD1bnNMhh1yg5iNAJI7kb7h3v0Byw0xIa4r99gtI6zoCX8oyC+5f2H1BwGlqYjoTpl59H3IE87b8UPuRpHwOKuKMYeM6Xwlk8Hba83NKpRTEBBzze9js61svKKpa+sqlxytL2PC0cZ+YyppGfWmce9rHI1HBz2ABNCpjaRyLqWGTKnepuAWCB/rULOKbu2y043mxXe8tdInfOkZhA/8Kvhg2P9/fXQUfbA72Pg1iSA2g/AeEGu+UqmyFT1+wr364LZV9+SDcNS5YNxqWkeUq+UFnF3EgCmH1W7JzlumezH9J0Y4npq5YWZZlfmUf7jA404a7UxSG9dRa6p+tiQlLhwLEeijXSrXoqmgDslA+BLuAqyDAMlUmnAhnacgpjwrbXddpw4eQXRjtXGHW4oc61zMBFkqzDULyk01y9+AiD+zTb5yTYPcI/sFllkoG7fCvYhaYkN5vEmS1K0PiRJ/1yUaEa10uVKURvuG1uCy9Cp9/UE2WOyS8YpwS8O8IOatsDLmhl8PGCo6C3p4EhVmkehB4jy3NQv4LdQ/gUJ/Mjix18X5Ww2zHE2DhYSWDnsyYD2xvM7OS2BMMb9BVYORTTyAjAeLkRwSUggzN0sPCJ1W0WwMfsWN4xPzUmU3aQNek6CB3r55RdpuvyNrWcONhsYkQVL/ojd//SwZcRc0lIoLCPCTUHLPBwEBoCMInJzIMYPoKWcOE2Iaj7rk6nnF11gjvNGGOHydSy7VvsjIz9to0dW9EQ9iGjJhtPDZvSrPLWxHlAN9Nx6mzUMQ9jc7vYCtRiRNa8W729g3ETNZXcM1dWVVpvymKhZY5tESI1qzWcmaYn7n9teO4NYPHtC6UMvgcLvkirfVrgegv8KLtZJxG7NHVCEUsCMPFEpTfoCQ0RD8yljoi/tZbIg0CEnojRIqam2A56AD4g5rFdJOhESm6TjO48BKHn2JGvNw4F6/f2cSun+mbbMztdb5F/ovuKvNwY60Xzb729NN94unj7l5blh8hs2Q3rbRhqf5TyoQr23gL77rs/vX3VgiyWqAWc3j8K/mUL9qUH7uUxMOnlugTFCmoI20MKnI6Ij3CwKLe0xkeC5/QH/9qvU7ppAyXGvZtAnYZP0QMyr5I19CQinjjYEyO68BuKgmfrEta4LLJ51twz0JWSHP6qUELcNGaNWNToivMibBgRWEZJsotvqmQRc946HQ3Hn9ulavtb1Ux1PocZLZIYzyKnw4FhAdzFUifBCBXzvQSq33coYq/RnkeVuSxE8AXeo9pltUn6fSb7meCtnLy8eYs9hxJHyP/c4xK67KNr/p0NLYBaBGmQ15kHIsZ/2oI0AMGJgd4w3HWou8HWZi/mLOmgRgTS4gUuMGDXQT2oMzo7UJpMLYnxauKs20wrlMjPauG1Hc3c6FX+n8KX8c6Kia7pkHowB3GHm2KR4slAgAG4P28zECC874BImLWjph+Gre2882znnRGGSThCT9+UU2tbnX0Yhkc5wlApt8FyfUj2IL1ge6GY8o3mymEIVXPblPhygP8oIuIoQ39gSFbGHJ6bFDG1WZHNGKIXvYUrpCgLepXm26AvOw0H6zQpzAtCsM1Efd01xQGcnbGxqu5fd0HWmYhPd1aehnLNQyrt/tyXTyNYHyuZGSR6wLLdxaDu2ronp1OugIKqoR+zc1P3PI6QbUea3ZmkYnzro2R872rZYtsrim7pcA8cWhbyCGd04Rvx0sr0tIDSeK+zfbR4ndmkyHvZS46uqq8oQfw4b9Mm8DOXNC1yET/OmRpUm0hlo5Oufl+gCAoWlors9uCr4+qmtgWg5RKGnov9Z9U4N2SUrHPt1DDEHgbjWEDVJnUA8SG3ApSUUJ8n2/nqPoYNuk6KQol1rhdzKc4eIldqo1aMcrtTXku12RHIjiDuEMHUbfAQtl0eYHrgzVUjtpDf5FvGIDXxOm5/9Amd/193O/n9nnaMlHC5y0OLuzwcIw6FWmUJPPv6z6NEnfd+EFH2UyE8xLt7n7S8b53ndMlFABB6hd9SyAnj6Kd9GwOMg1uMK1IiLBK/cVJ6CFu0NLfG1Uab8Wwlt+FD97EbvSRHgLnvYFgKyC2O57YT0O1jxnMEmM7x6NPISkrlAHAFKuQtXotEdoc8EKd8joMO94ppQ0BPcBmA1NFRSuov6pWhvA9rkTF07Qrop22hDgFtqpYPHtXygNw+BTaR5QvpP3Iu70tXE6YhQH8Pzhdjh24BC2h314d0gAdbB3h4Ph3g3tcZbgujM/n46yocNBJHvVgeoVjwm4flFrBfcreTRD5XMfjlyrAFd2lqJHa3kl9IXuHwicfxCnN2vh3WpZ8cxUL26D0WsONZyaHh2mM8hq3sG6MFDdnLY5mL0x6pi/6cE7STE1+ZT5sSSpSl4Qh25+g3CEIfvCi6nJdFjac6xfzer9PQ4fQhF4DqHbZRU2UUP8Tdmg2oYdMh+tHnK6jYuu1AzOmPry715QJqJLKE7NVFkOdRZc33tH5f14LtWh4TpXOhB95sC/r2C/bFMDyac/KpzUHT3KhhCM5yqB5/ciWAUdSSBRy8JQ9aEoDo+pS9//M3f56wixKPJlZJQ54Zcwxlkd/jOTymkanZJfycr3jNy6ssoqPREfdm4mN/NCMvbdeUuO4SAIcNZ870JhEqNfsaUrMJthvadlSyXicxVY7EQ8YfpeOPXpoYs3Eu+KngwWa9SB9NXWXOGSjYQuhHVz1xKDTDmbeirypIGrvuqjYzTgTG3EI0VMw+7RMdb0NRE3qC0RvtzffOWGEjnEtAxHwE17uuAxxcH3sSzMnlKtTY5TqKhdxU5V2z8jONOmJfO3bLW7UedWhfrFTzTHeboP818CqRoalEz4WU2+1G9NeKuPOfexizJiNZI+BtKP2J4dnZ2DiXToomQ4xm83hbZEsaCM2vQAtJT6oY450VO9GKSBqht3dQjPmJvPC8GK5r5dzWJ8rQcyxAjGB0+NOclq3z8NQiqg1XcugopRX0kzb28nEGEt+mc4DX5Z3Or+na//Z6aviIyVLGEjrKoSesYN67TXfzfLtIFyCAdmlNx/nGbVqUBTHyuOloOLRd2g3q19Pxfs/zO87Fpb7Ep1AzsVLA3zBflntAnOTs//z7v7Or/BrnM+MD56/EhNRcDPl+t8qgJz4dPJZ7dfE3tilRvBLPXCZ5XsvLeuh75s5lNX9G8x+cdCBmwnKQvdQIE5/Z+nQ7OCRIdzRM+LPlf9TI5W/UflrtBoOBHfyhx/MaHeU0KIrtTDabHD2Q2mfep3wKKBB+wmkDckXopzo7qduY5tkGOHLNMc8Af9+0fFJObc+cZiSNX715w36yCMI8prMx6w95cOpQ7I6mfKIqJ/tDla6TrOB3FjXd2lpCrLUERcJhK+ReQ/rKGyN+DYj74B4R4+1OobOAOsWtNPjzU8qvl+ZlubGJpoDu74V+o3oM2an1DBx2NPgC5cfoS7vHD0AACXGeU+T6KskESIHREMDI1DpnzL3ujMsOqxbj/RK8AH4NvCugf/PrULF0zg5xjJTUZB9HxE4CGk/Y7mhrdOTxcaJ0wPEjj1fbgqknGNGThmQvDrGUKc8qogbUtxChErp45InoUoT2wFqYDdu863BH7ek4sTx+NDsdn/p6NtbROEwdLKrkjvdFh7Nd8HN7iU6J8U7ZWDldF8ejqS+btFa9BfGJ+NA9OHuRnECaVa6T+oPt62mF+AH/qWXfZqQKJpnwjuKklfXlOJZPR562ZGlH0HAhNWWB9+oGURf7w5RRhyH7J/HmqynvG9742xHueTu5ubAxvf5qao6y1T4cgPgUIfqtQsLkP03Z3/nAT9pMQOCaBA7+nrU5gahCUoZXsQ2S5ANiZJ0VgYKoDMJIs+zQZebU8A8+Ni6kiJuohruMVCdXE4QglBDzjSf7jWbb/Sl1bIk9XfoHfybCy22BIW38Pk4LeO/bBBQcMtaFnEmLcnuz4kpOjZYNkY1SaDAIFGT+gPXasC65fFJ1bzGGidI6ZQX6UepU63iDXlcoVitmSaBUhpIacRmn7BWlMeOWJcr8hF2wHaO4vSrbaVuFDEauwrsOjVPDJkNjNwDRptjHIktu8KYeNyLZv5ghN/8izP73yOTsCm//Eg7wInCTFkFo/NRODu0OsIH6Ts11s5+3Ccq2AJufiZm077lQMwqTxOGLkXHw9PvH98YpPMVhtUdvpYkSEE88FrqBMRrcOUIMjWE4jbTRbHV56S5/txPgRK7999rwnfBl1+6tpIrRejWXnIe9GT/Fg0EKj/I8GDgkW91Eypnuy7P4pm2Pi2f3dGYOThp0I/M2K2WGMax3Tco8Ln+ZoEGkuoA9l62hG8vwVwnf8BmBkDqY9kfakD0VPdGfF8yEfdDQRqjYzvQsqNVBr4V/hSx/SOS6N6BapePuh7984ZCTa4gttfzZnDfdS60dH85cQ73y+yKt8wNuIjW9/qjDYWTWcEHHebpsukkVWqMvMjBARLAss33z6moTeihdjoKyz+wdhoI2OdS5UbOry6p72b3ABUH0Ncb6ethH7dsn7SK5g/RFskWXG4dvjBOvpzMyI6ta/vbWDlSn5K2q6aaeEgMxowmrNOfxDFqkkU6WwJyBL2J8XpWChrHFlKJlyYJSXI0PXQfQ47a4lD6RM1jjUBem9m/U5N947mnKu0TucVCG0PbAC8vJMmVJvlklLChKivDJUtSLijQFDUevIGZYUG5LOkh0g5IRk7CesS07vVRG644HlaD4YKB0AKgNQ8vT6U/daPeRdXMGfw/+eLuFc6CRVusY0yB35x0l75Vvvg7P5OeUbbPCnUaLbVHDmSfYH7Ug+7XC1LkzD76qUz2fAfm5CQLRKP50grHsOyQkEBd4eEVAMJ9Wg8fiexQDew07mQnwBxpe2L0xXR3Cc4JyhJRRHbXTvTp4NDFi5HPbExInLGbosV/hrV+10Sf4jnQQRsWkfrhJeV/R9RssHYbs+p4lPL0J0Ckau2A1FVkDRgYLRL993KoW5mPavFNahHNsaOJOFlaq8NSkPhqqtX0kQByzflupty08nDpnkQqs2HvOQbDs0RAPvCJmjYJ34uEcS9Rv3LXumWfAIe0NZZFM1sOEg1WaA47TpEJGWEkHIq3OqaE3EWvnNqfjMsqfNItT99QqFpDgD57NyKmceq59UXXALx15Y7MzBeHEd6Zly1GuEeCRQdzg/a5GqqT0EK9vljHm84sogV4sRYBKEipONdNbcpXRSQfLr+nUFA8aUFGN9hyZGF5lqPrFcMhT0qabck47fDQc4ju6hZNXPPvpywgjHGMV8I4H3SP+7utLqwo/o+OvPjcEEiygNRtbVIlbjcbEjIstOvINP4MCw8ZB8U+ivFok6x8CfcljsJGp3GqMqKDhi4kYlzbUR1Smnd9XGXxd1lmRvioK4PtAj28uA9V7xN7TBWGNtYjhuPE2scQauquHY4N4zE6tj7aYcImLwiL97ncdLY8d7g9Jtb5MQThUTR3YFKy7s9+/j4d0OKb2rU7XXzfIBK/TJd4eo3R/FHUBwF0Q623eoDTjEFblHVvjaf975Y6pxU1nb3uJRCCdL+mTHYBNMCPZm0tdz+Y+cUIu+auZpT52lfAN2lEyr9oFdNkQVxglAH4BJ+CHLnrlw/axyx4yV5qxilxAWvdGcbSqHnd/yp8px1Ja7WNYYhz8hJLONjTncM8op+Kvw0D0g3NaQbaud0O7EYH8FHq+KumjDwcQw6tfQEXRRLlPPdXMy5/ol8dDKSBxvOCMh8GBHnykY18kXB9+3cGKX1fQ0eyYiset4qm9/Pr0XRgjApjv8N2KRtGM0fVD7A0/Gc54xJ8I+Qtbd0LtkbUt02ODV0YTsydx4tLubl+H3i7VSOIGlEdsZ0tWPzZqrd+6YSuOqqzVMbun0A1rcTmSdNLjs7qN3aGHeEdijsLpGxQeTz0xLm8n7ojwnRrV6bHI2TekNjaAKViahPdIQ8YQWTXPJDl7rjpqQpfedl+noKp0dgf8X2WO6r53sacPR5BYeJ1X3sV2pwrjO5MwfOMXutVBlIl6AmGeqKn9qJI58wpYZgTr+95DhZy0d5FcYHY+HsmNIlJo7wNPDjeh4pEK26bGY/aG0lzospVwebRADa6T+QdK8eF4EeQ3+LZNltcgWrIND79GZ1bcrUPi+RFWoeCyjvGgguR0p1UzUXhihEy+BuG2tVOJbfJmgB/gCyQ6QqtolyfXwMV7r1Hr6Nll96LsDbQ0ik6pjFwGQQ+YkNOqyRooWPY+yg6v+qPZZDBOPzkV6+Q2XWY3UPV7GF/94qO89PzpBeF7U7ig5/kycN+Udeq742VMW3PJp0z9hr199/pJs9f9PgkDvPmzYUFz5sdj4VLa1nT0+lRSEN0/ARnY8rmRwbnp43EBvPQXYYH3+wQkzKun4cD2DhwTHsnm26ZcLqd7rf2Tzkg0O0TSslqmLUdDd0DlF25A5S/NICC9I+nP6BE0HEn/i6IgN6vEECAUIhhXCFQGfRotvgON/5Z7mexmmSqw2+qm6Y7nlzFetRqhsXhi6uyLbN4E3DCc/pZs6byafgHWbMTeD6djfGPYuP8zjDpbj6gytB7L1i+d1r+1WhthJqbSNLW+h8N1jKn9NRutMLQrk5zlxoadzAEL2xc6GhsCbxjL7FqtANvncAmZwExa8EdjWjXsWExMcNtaXy8QLxnMvCPSGOg2f406U+Oh21D0XuY/9j+jMcjMG573p2goB0/9PJCbsknyuNs7s67TdgEeLXoLBDttF9A1bH+RBEaanSp9LrxxNiRVXYex/CLY9K1bYOvLGwqvw+RjVdKkgUm2jjKuvLbQ6KrHn3rOqXoli/PKKXo/5EUD9EH33g97kQDoOJU41zHrilfQYNipFh/vpZ2a7tnn8Xm65ofh9nSL0PP5fuh7L9yZ7XzUgg2Lv875n/XUiq/0+RMF0tEDVc+nPZvCemGHtWdf8nWIf3+pzAvrqyJjZbqBCC4lMxH5K5E97StQG7ejXNir+6bou0q4p77PcMQvkzcl4AGEZPu6ZmvlPJH3Wk8J/bnJuz29LdHHP2P7ryf7Etp4nbdT7rqdHue4bemA9gvTszvt8useFG90dQMfDjjCW2jqcHMifNPT6QV0KjwS4pCC2IxKEUrp29bpugSSER+cShd0Qoj66dnZS/8Mj3SjukPgBxzPMoI9FwTdm45PuSr4ZJfuo52zfoRRIAPt2wyPSW8Yfej6mPUF/kQNfymCNZ/jiRVliOD+8M3HjJA+Z/NMw7SyefuCsY6KaTxq+Lgaz4VkW6DQ4K14zCdFYvpJqh66uWxHrVy2Yi78+oX/UoQP9NuhzQnqYUhHJd7KLQkZ6M3xdii+vRXacRkde+R7gpE5V/UsKfsor3UrYdDUee7OHDTtKghtD7v4KLtHEnvlqqsUPMLN/6tOxjyj2D8lW2U5zuvedurv72Of0mNnhe1KsODv32PGP3ENxWAsZnXu+hHYmbs5zvXZB1DzuXvooynj3D7cMJF+3jmVs27EdU7CPEJoVTo9fAxB1fadRfyS84jWMcTBowjvccRRq2qb8tYBjcpl03WwZVj06pMy/NXxbaUB7wCQn0s6AMVwHUgA88ptZh54yZzh8ypU5B922y8mYPXyGPiqstnNc7knDJ+n8lFw18Gv66RoucX+MT0VLXv8g7bojI/Teq7Z6Q0JNl/6kKIKFHi/zfVhz0e5DDDbQgCSuWyCx3z5ygdIJ/8OPqCO4WFfXd4Zbx/LLMevovLkhmwzYRs7dVXEOtiUJyUAX+Bp7nHA7GFXz+b/OeAD2ucH2u8LOugPci4btrwJnX6gZe+S8o58rD5FFORLQYPs44dPvf+SHof/D9wQj3QZ/Ld9/ysY4v8PjOX/tmj/kSza/zDr878txf8qluJ+dWKvIXgoLu1XsQePsgn3h6kdaQd2GV8HTMBjmu21/vYB8Bt+T7X3jrD5/oNMPYw9ItpKn8m241nabtOY4q8WCQqpAmgtYjysylE7VaQW1gxbRV1hWk6olgDdLvZGatnRWji4dtme6KyP2OKTE5zlD9Byg7SMtF0SQ/a+iFhPvwB7s/cen5gTlqlbG9sDKssn2Rh0p/7bd68PN5fbxIBBrySgC6XysD8m1X6Yxs6BluIJgQDT7WxkbQ6oq56x4Vvz2zUChE6SkjYx3ScCNr4GHkxczvwmk6v8WIujFCHnKqlHJ7IrtPSjyEn1Y5i47gcelfLk050Y3tulSDGRnPbtn76lC97W50M68gBbY3hrfrXBDo1wLFQrXbAdXW0lN5ZZTrA/DltcC8ZMWpgUMDAWRqwHv6OItyZAwsadQYnFhFnfSgMbyXmTxFBnmZeJ8S6nOphEK7Kv3IuaeIfQ+lLQlnbHDX7mt8zVbUgeVYZfFqBW/vyNX4P1vG1Stt7G6Bx6ffXXH4sZAKQEi/lNWWXNas1efoaWCibAw3RbeCF/mycseP3zgAUvX5pOjbtVWqXsB0DJQ4gflgpgggx+os35Fq1QTMkY5GlSFekiRDIwMjJ+r+S3euXhxQUj/NhUQavCymqRViwoMAnWKByYWGYCeVazV4tkQ3HByFFxyuh7aFZoLOvGuae/N0nTwPbof19mdQ0oUZoHCzbJhq7xzcv1mpLSjzDr0tgYDa2lfzzz+bZK5vdteFu8ITgy5yQWnfFVt8B8t6TPZkXyM5x0dxtXGLOTgToDO61c8iUHQQFVgBlDldBOTfmXOl1uc/rsGmGXf7UiqZsKONc6A4zl9/wzjMYSXvIe3fXTwWnYaVnpEXmpEvos2FdgtnuSaJmftC/UB+1xwXtWNhlc83Z6xzYMrCehbMqavmluQ8rJWwKWOlqY4/AQwFzmkgeAfO1ZkNSaIsSNENgHtLRhz9J7LpNFtoXatB1Hwx8/9olcXgRjQEkefhK7bmTePpOVB0PK1Mhb4DXvMb6BdmfQ0upFfD+tvU/lpjbBI9/7ASpO5IV6Yiri98SXmo9/yeDiQny8oJWa71QwZ0zrAdsuTxG1xjfNuY7oBJdunt3eP2X0iUmeOwJvnmDiWAaWVZ0WOqmqO4yrITm7ZvSBe55MDpF3xh5Cb5pCfqUlRXSg1Gshg99W58kc4XcAXYTYh0dkukgskSc3Al24KJv7ZoVTcDJUGGJODhldoK2vnp6y90CWi6yG5YERg5BFUpVs3gz5fgdDBpqoKN9afxT+WOBjn54x9yM0jT/+NB19+vFjkeMHbi7TgBqw9MePGdtk7KcXARE05t6Ad33+Eqg2/2RoE0VOWZ2BgDW1Y66IuNCjtsZCWdJgFxQIlw/JnN8rpLZ6IhzBP0FL6PWM+qXhmEYExq8ynmRrwighNlZV9Y3KKDZ+0sclQJhF7ktST7lQm9VAwShsxo3fI6HPBshvUmLleSn8ZnTnCl7AlhEv3MudCbndWq37dqe5CahVZsN8h1fnOBsJfiBmccb7CUlXsr/jm0pkFb8h7oMci1VAGsmPhYlZpThELLsBlRwlb6MJL3ugG2D9FPkp5XMxG78vWbbeVPhdXszxkuWoP+MC8C8CF5FKqQHiaS7UG+5m7ZObdWDydqk6OTsLxVPxG5kUhbNeeCh4jhT9UyYfC965e/+dIQUCOgvkEjPkmVylpKGRw17MFri/698zdG+QvZwV+GWGG35vabHlWYfRD1Mul7agfheL5LCkUATvaGValxdFLX/GYy7FvsX2sBjfY5htR5ZKIvfeO5jWHDQUkIxFX80leDf9yLv5FPqyUpKYf4/q1Twp2Iq8BTg0ZIupuvnH9YwCNKt70hRecHGGLzEtVXJzAyZIDf0NegcOzMivDFQpt13O01IVdJAJa8hf4gIbcpLW9kyXVa03iam2WPlMRHfnui5Hx760ceYA+f7JQAgUoSO/cQNWirFxiYMPzhfpuBpPMM/20IMYMVY/Mb6VSmqetm24bAZZ12XfXBy0b/IY3Wdjj0XDS14+1tbhI8c9G2dgfYH8g3p3yvTROZFISXmPudyv3oARd0XgZhFVnB0whoS+C+gZRfhdevhx0TaKgN7T+YB9OUDl/ngNWADnCeddvd0/PW6+TtiVNaLZkY0RJ2SAyTesRDsDpGCnfWAzmOBttkvVt5EwtStmhEoXv2d3wITvsga/8JrnokKNnIqykpXFPBXp2QaGRl132S00UJ4cQwwNeqW0/OIqDo20LzA46DIdLjymA6f3K1h/7gkwBMoP6Wcwdkzdi13goqNzZWSYLBNmUBDemQGKpDxk6y0mJJtSG8d5yd69f/09DIOy7MCOFqKIOBy3vvi3XdbbMXOcl2Tw1fSZwBGOJKKKZAEmbJVuKxhMNifJkYzZdcKT5RHWZGcIXVLnqM8p1J3xopxYt+x4Z9arufIO2O8T/NbG6AX2woI6o2TFpEazuuRfY3h9BXX+OkNi+fso7ACq85vyvo2TEgQtlOIOl8uItHDADaZbwtQ6PEcQ/RtJuc43Tmh/Q3zUaRd6BGBPMIWFaggyepXOP/B5vkWViJIdKyEOFJ9UbGhe9ef4Amb+wkIlrv3+SY7lJMdHTNKAO0h3DXrigWGM8Dvx41nopc/xhNulcpUVScIWVomiE+mV8NLp9uYmrYFWgWARdXwpacLjM4D4AqEpRsneAc+3nCHJiKqSoMMBvKDuaZ3G+hDtqaRBaBMy6DgKecSaPAn61WiGC3I15nyDr45tms8xy/cdKPgrwO8d56Am49OZOSkpZnMftpxC0zabU2v+cmJswZjceayYvhwMLpgbzlZom+ZlBHy1HQCGnpqy8avRvN+ENJKgALsM6CH+WPTHn17wHyPpzRj/zs2rojbrVYEn6DOxZU99KmXn3l32VD9i4yYFBb5QRtG48GbjEMMV5/ZElTSG8YxoU4zHF93m9zjZgEfmZ9D0xe57fxuhzu2jxkJSY1xoanx5HDV6FpLU0KxeoiBJhU6APKFD7eoMGtKEaB0TGHfu003D/izg0HJNusb1zH3vjx5UBHVy0oGiTloXrFcegm5dM2F/x2Ayck0qhQneNysyiktttiIaqhIMBLB8jHMT195Zb1FBUfbHLqSNvKNMndsZdyhRoBmpt8PBQG6RjumIdJnrrbvt92z5C1AxfZb3o7kDqvBMmlx8JFRVGN2ygbTTZbvQe3WTd6Uwo3b4ubHD+3t2uNFehnolviXm6Hrejeu/iKmJTfZpu5KVkTeiZb5gwaIqN7TurYTXKJkmM9WjnXnDtQxFWk5u/hkSQQoox1F74UlhhlS5cCOMHeEiu1V2aGTsbXHG2Gmy2l9r8IJT41QxCU6F8Gii74aNtHIAsCi+mkQL0grc2q21EgZQ+3iSHMh8cXYGMjGObnc4w97hzNm+MM5dR449PX3+qkVy/PVgvtm6mV7wCL115N3gaS6mAgHYH3vUd6xf9iZmyDXBHOjSIPx0YscT8S5UhYgte2+xqRUUARQ12DTm2b7uYi82je8r0JXq7pnIjHlQbc8I0GG7ieliISh+/hMNY2wIzsCNMc+rNuJmv2B6GOYQI6tPqjRepUkDwxT0Jz5lFcmvhKmgaBUM3Ype1QnyIkZhLObnHcWX1PK0CIxM//zzZ6aBKZ04N3l5jZYwsBgyapId8Zx5mcO/ZKzqr6RDJelDxQzsdlb0Ab4K3REaRaEBKNkpQMmuBQhedQHCotAekfiUlj0ifNkxIl3kjEgCskbEa3tHpIsMfp3dgKzioeoUSrS9xsWv0RyC/6GYJw5Fo2g0DEM3z/V7EDz0QQZsNTEir4NL9E7ax2LZekRWJHTHT8JgSPWqvAv8ySw5CquU8p0GKFkL96IF2cPNNDhMlU7DsspusmLaA40zrXqRk4lnk86baQ/DVZwiXMCpoiu3LNlNFal4QmiMeVM2Boro4sxBoKvXWVnGlu26q8hMYPc9a3kHtDWukypA9ONiT3XDR6znd4DH9nqOjfUcHVpPIsB/0PVEsF3rSR95717PkW89CV29zsr71nN0/HqOrfUctdfzDeLDu6I8Vax/m47VNh0d3qYGu/tPtldH7l79E53Uof0JGpV/y44Ob9nRY7bs2NiyI++WPbjE3p07HhtLPDpqif8zbt+Ru33tJfbu4tHhXTx6zC4ej60lNncxj/y9WTVxntyDYh0cnZZRaG3xR5rXgBAb4On+JgeTPfiMfRaxz+LPwk+PStzo1RCFZvgIra9T6xi1tY4vDKUDV/oWL9xqpU6kaG6pd/I+sszlYdXQLKt9mxopxuwDv6QlP/J2qA+zxt4+xDwMVdA3Dyw+Yh5mtfY8DC3RNw9vH+15+PvgggcIfY4f4Ksc9HhV4nbjZJdnBTAUc3Eja6nxG1PmskTWIoWR0P6nrFeli55neMZWJfHcUUfuVZMHdFTl3IL+Zees1yUMfKghVuXjre3GLmp4U/PJRI0oNZ4OoGZ0BGpGx6Nm1IEal4keurLwrBzr/wIhE7fJ'
_NN4MGF_FULL_CONFIG = {'joint_rounds': [{'epochs': 100000, 'lr': 0.001, 'T0': 100000, 'eta_min': 0.0001}, {'epochs': 100000, 'lr': 0.0001, 'T0': 100000, 'eta_min': 1e-05}, {'epochs': 100000, 'lr': 1e-05, 'T0': 100000, 'eta_min': 1e-06}], 'train_freq': 2, 'batch_size': 16384, 'max_grad_sample': 1024, 'max_boundary_grad_sample': 128}

EXPECTED_HCN_SHA256 = '3896e23997cfc939ee4871833591a3714e0a79791086ad700fea56240a5b2018'
EXPECTED_NN4MGF_SHA256 = '46d123430d459a2dfee064ace4742bdf213fde024c49bdd4da317f1a549364e7'
EXPECTED_CFG_SHA256 = '172d06a525894b8f1889cc863016b7750343403a54caf35c447b368824333d53'

def _decode_blob(b64):
    return zlib.decompress(base64.b64decode(b64.encode()))

HCN_NOTEBOOK_BYTES = _decode_blob(_HCN_B64)
NN4MGF_SOURCE_BYTES = _decode_blob(_NN4MGF_B64)

assert hashlib.sha256(HCN_NOTEBOOK_BYTES).hexdigest() == EXPECTED_HCN_SHA256
assert hashlib.sha256(NN4MGF_SOURCE_BYTES).hexdigest() == EXPECTED_NN4MGF_SHA256
print("Embedded source hashes verified.")


In [ ]:
# =========================
# 4. 载入 HCN / NN4MGF 原始实现
# =========================

def load_hcn_30d():
    """复刻仓库 benchmarks/run.py 的 load_hcn(30) 逻辑。"""
    nb_obj = json.loads(HCN_NOTEBOOK_BYTES.decode("utf-8"))
    module = types.ModuleType("hcn_benchmark_source")
    module.__file__ = "embedded:d30_skew_symmetric_product_form.ipynb"
    ns = module.__dict__
    sys.modules[module.__name__] = module
    setup_end = 10
    for i, cell in enumerate(nb_obj["cells"]):
        if cell["cell_type"] != "code":
            continue
        tree = ast.parse("".join(cell["source"]))
        if i > setup_end:
            tree.body = [n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.ClassDef))]
        exec(compile(tree, f"embedded_hcn:cell{i}", "exec"), ns)
    for key, value in list(ns.items()):
        if isinstance(value, ns["torch"].nn.Module):
            del ns[key]
    import gc
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return ns


def load_nn4mgf():
    module = types.ModuleType("nn4mgf_pinned")
    module.__file__ = "embedded:fit_mgf.py"
    sys.modules[module.__name__] = module
    exec(compile(NN4MGF_SOURCE_BYTES.decode("utf-8"), module.__file__, "exec"), module.__dict__)
    return module

print("Loaders ready.")


In [ ]:
# =========================
# 5. 计时器与输出工具
# =========================

def cuda_sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

class Meter:
    def __init__(self, warmup=0):
        self.warmup = int(warmup)
        self.times = []
        self.seen = 0
    def iterations(self, values, **kwargs):
        for i in values:
            cuda_sync()
            if self.seen == self.warmup and torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            start = time.perf_counter()
            yield i
            cuda_sync()
            elapsed = time.perf_counter() - start
            if self.seen >= self.warmup:
                self.times.append(elapsed)
            self.seen += 1
            if (i + 1) % 100 == 0:
                print(f"Completed {i+1} updates", flush=True)
    def result(self, samples_per_update):
        total = sum(self.times)
        return {
            "measured_updates": len(self.times),
            "measured_update_seconds": total,
            "mean_update_seconds": statistics.mean(self.times),
            "median_update_seconds": statistics.median(self.times),
            "base_inputs_per_second": len(self.times) * samples_per_update / total,
            "peak_allocated_bytes": torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
            "peak_reserved_bytes": torch.cuda.max_memory_reserved() if torch.cuda.is_available() else None,
        }

def real_scalar_params(model):
    return sum(p.numel() * (2 if p.is_complex() else 1) for p in model.parameters())

def fresh_output_dir(method):
    root = Path(OUTPUT_ROOT); root.mkdir(exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = root / f"{MODE}_{method}_d30_{stamp}"
    out.mkdir(parents=True, exist_ok=False)
    return out

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, allow_nan=False), encoding="utf-8")


In [ ]:
# =========================
# 6. HCN 30D benchmark
# =========================

def run_hcn_30d():
    out = fresh_output_dir("hcn")
    ns = load_hcn_30d()
    seed = 12345
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    model = ns["build_model"](seed)
    config = dict(ns["TRAIN_CONFIG"])

    if MODE == "smoke":
        config["batch_bar"] = 4; config["batch_bdary"] = 4; config["batch_mono"] = 4
        steps, warmup = 2, 0
    elif MODE == "profile":
        steps, warmup = PROFILE_WARMUP + PROFILE_MEASURED_UPDATES, PROFILE_WARMUP
    else:
        steps, warmup = 50_000, 0

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr_stage1"])
    samples_per_update = config["batch_bar"] + config["batch_bdary"] + config["batch_mono"]
    meter = Meter(warmup=warmup)
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); cuda_sync()
        if warmup == 0: torch.cuda.reset_peak_memory_stats()

    started = time.perf_counter()
    for i in meter.iterations(range(steps)):
        if i == ns["LR_SWITCH_AFTER_STEP"]:
            for group in optimizer.param_groups: group["lr"] = config["lr_stage2"]
        ns["run_sequential_backward_training_step"](model, optimizer, ns["PARAMS"], config)
    cuda_sync(); outer_seconds = time.perf_counter() - started

    report = {
        "method": "HCN", "dimension": 30, "mode": MODE, "seed": seed,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "precision": str(next(model.parameters()).dtype),
        "executed_updates_including_warmup": steps, "warmup_updates": warmup,
        "base_inputs_per_update": samples_per_update,
        "parameter_real_scalars": real_scalar_params(model),
        "outer_call_seconds": outer_seconds,
        **meter.result(samples_per_update),
    }
    if MODE == "full":
        report["total_updates"] = 50_000
        report["total_sampled_base_inputs"] = 50_000 * samples_per_update
        torch.save({"model_state_dict": model.state_dict(), "report": report}, out / "endpoint.pt")
    elif MODE == "profile":
        report["total_updates"] = PROFILE_MEASURED_UPDATES
        report["projected_full_training_seconds_from_profile"] = report["mean_update_seconds"] * 50_000
    else:
        report["total_updates"] = steps
    save_json(out / "report.json", report)
    print(json.dumps(report, indent=2))
    return report, out, model, optimizer, ns

HCN_RESULT = run_hcn_30d() if RUN_HCN else None


## 7. 清空显存后运行 NN4MGF

两种方法顺序运行在同一张 GPU 上。HCN 结束后先释放模型和 optimizer，避免污染 NN4MGF 的 peak-memory 统计。

In [ ]:
if HCN_RESULT is not None:
    hcn_report, hcn_out, hcn_model, hcn_optimizer, hcn_ns = HCN_RESULT
    del hcn_model, hcn_optimizer, hcn_ns
    HCN_RESULT = (hcn_report, hcn_out)
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); cuda_sync()
        print("GPU allocated after HCN cleanup GiB:", torch.cuda.memory_allocated()/1024**3)


In [ ]:
# =========================
# 8. NN4MGF 30D benchmark
# =========================

def run_nn4mgf_30d():
    out = fresh_output_dir("nn4mgf")
    mod = load_nn4mgf()
    d = 30
    R = torch.eye(d, dtype=torch.float64)
    for i in range(1, d): R[i, i-1] = -1.0
    trainer = mod.MGFTrainer(d, -torch.ones(d), R + R.T, R, hidden_dim=64, dir="benchmark", x_min=-5, x_max=.5, y_min=-5, y_max=5)
    model = trainer.model

    if MODE == "smoke":
        batch, warmup, total_epochs = 64, 0, 2
        joint_rounds = [dict(epochs=2, lr=1e-3, T0=100000, eta_min=1e-4)]
        max_grad, max_bgrad = 4, 4
    elif MODE == "profile":
        batch, warmup = 16_384, PROFILE_WARMUP
        total_epochs = PROFILE_WARMUP + PROFILE_MEASURED_UPDATES
        joint_rounds = [dict(epochs=total_epochs, lr=1e-3, T0=100000, eta_min=1e-4)]
        max_grad, max_bgrad = 1024, 128
    else:
        batch, warmup, total_epochs = 16_384, 0, 300_000
        joint_rounds = _NN4MGF_FULL_CONFIG["joint_rounds"]
        max_grad, max_bgrad = 1024, 128

    config = dict(
        lb=-5, ub=.5, imag_lb=-5, imag_ub=5, excluded_boxes=[], full_gradient=False, theta_eval=None,
        batch_size=batch, max_grad_sample=max_grad, max_boundary_grad_sample=max_bgrad, train_freq=2,
        joint_rounds=joint_rounds, individual_rounds=None,
        lam_monotone=10., lam_CR=10., lam_growth=0., lam_zero_anchor=.1,
        lam_boundary_consistent=10., anchor_set=None,
    )
    samples_per_update = config["batch_size"] * config["train_freq"]
    meter = Meter(warmup=warmup)
    mod.tqdm = meter.iterations

    old_cwd = Path.cwd(); os.chdir(out); Path("Plots/benchmark").mkdir(parents=True, exist_ok=True)
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); cuda_sync()
            if warmup == 0: torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter(); trainer.train(**config); cuda_sync()
        outer_seconds = time.perf_counter() - started
    finally:
        os.chdir(old_cwd)

    report = {
        "method": "NN4MGF", "dimension": 30, "mode": MODE, "seed": None,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "precision": str(next(model.parameters()).dtype),
        "executed_updates_including_warmup": total_epochs, "warmup_updates": warmup,
        "base_inputs_per_update": samples_per_update,
        "parameter_real_scalars": real_scalar_params(model),
        "outer_call_seconds": outer_seconds,
        **meter.result(samples_per_update),
    }
    if MODE == "full":
        report["total_updates"] = 300_000
        report["total_sampled_base_inputs"] = 300_000 * samples_per_update
        torch.save({"model_state_dict": model.state_dict(), "report": report}, out / "endpoint.pt")
    elif MODE == "profile":
        report["total_updates"] = PROFILE_MEASURED_UPDATES
        report["projected_full_training_seconds_from_profile"] = report["mean_update_seconds"] * 300_000
    else:
        report["total_updates"] = total_epochs
    save_json(out / "report.json", report)
    print(json.dumps(report, indent=2))
    return report, out, model, trainer, mod

NN_RESULT = run_nn4mgf_30d() if RUN_NN4MGF else None


In [ ]:
# =========================
# 9. 汇总：runtime / memory / speedup
# =========================
reports = []
if HCN_RESULT is not None: reports.append(HCN_RESULT[0])
if NN_RESULT is not None: reports.append(NN_RESULT[0])

summary_rows = []
for r in reports:
    summary_rows.append({
        "method": r["method"], "mode": r["mode"], "measured_updates": r["measured_updates"],
        "mean_update_seconds": r["mean_update_seconds"], "median_update_seconds": r["median_update_seconds"],
        "measured_update_seconds": r["measured_update_seconds"],
        "peak_allocated_GiB": None if r["peak_allocated_bytes"] is None else r["peak_allocated_bytes"] / 1024**3,
        "peak_reserved_GiB": None if r["peak_reserved_bytes"] is None else r["peak_reserved_bytes"] / 1024**3,
        "base_inputs_per_update": r["base_inputs_per_update"], "base_inputs_per_second": r["base_inputs_per_second"],
        "projected_full_hours": r.get("projected_full_training_seconds_from_profile", 0.0) / 3600 if MODE == "profile" else None,
    })

df = pd.DataFrame(summary_rows); display(df)

if len(reports) == 2:
    h = next(r for r in reports if r["method"] == "HCN")
    n = next(r for r in reports if r["method"] == "NN4MGF")
    if MODE == "profile":
        h_full = h["mean_update_seconds"] * 50_000
        n_full = n["mean_update_seconds"] * 300_000
        print(f"Profile-extrapolated HCN full time: {h_full/3600:.3f} h")
        print(f"Profile-extrapolated NN4MGF full time: {n_full/3600:.3f} h")
        print(f"Profile-extrapolated runtime ratio NN4MGF / HCN: {n_full/h_full:.2f}x")
        print("注意：profile 只是预算估计；profile 只能推算配置总步数的时间，不能当作实际 full-run runtime。")
    elif MODE == "full":
        ratio = n["measured_update_seconds"] / h["measured_update_seconds"]
        print(f"FULL runtime ratio NN4MGF / HCN = {ratio:.2f}x")
        if h["peak_allocated_bytes"] and n["peak_allocated_bytes"]:
            print(f"Peak allocated memory ratio NN4MGF / HCN = {n['peak_allocated_bytes']/h['peak_allocated_bytes']:.2f}x")

summary_path = Path(OUTPUT_ROOT) / f"summary_{MODE}.csv"
df.to_csv(summary_path, index=False)
print("Saved:", summary_path)


## 10. 最后下载什么

跑完以后，把整个 `benchmark_30d_same_gpu/` 文件夹下载回来即可。每个方法都会留下 `report.json`；full 模式还会保存 `endpoint.pt`。

- **runtime 主指标**：`measured_update_seconds`。
- **显存主指标**：`peak_allocated_bytes`，同时保留 `peak_reserved_bytes`。
- checkpoint 保存和 tail evaluation **不计入** runtime。
- 这个 notebook 故意不做 raw/clipped tail evaluation，因为它们不影响训练 runtime；最终精度可以回你自己的 4080 上单独评估。
